In [1]:
import sqlalchemy
import pandas as pd
from local_secrets import username,password

In [2]:
from sqlalchemy.engine import URL
connection_url = URL.create(
    "mssql+pyodbc",
    username=username,
    password=password,
    host="main-workspace-test.sql.azuresynapse.net",
    port=1433,
    database="SQLPOOL1",
    query={
        "driver": "ODBC Driver 17 for SQL Server",
    },
)
# engine = sqlalchemy.create_engine('mssql://localhost/balta_bronze?trusted_connection=yes')
connection_url
engine = sqlalchemy.create_engine(connection_url)

In [17]:
# 3.b.1)Python skripts kurš veic unikalitātes identifikāciju polišu piedāvājumu datos (offers). Izveidot šādas dimensijas, pēc rezultāta ieguves izanalizēt vai izveidotais risinājums ir efektīvs:
# UniqeDay – Pēdējieunikālie piedāvājumi dienas ietvaros (24h)

with engine.connect() as connection:
    df = pd.read_sql(f"""SELECT id_offer
      ,id_customer
      ,offer_product_code
      ,offer_product_variant_name
      ,dt_offer_date
      ,amt_offer_premium
      ,offer_sales_source_name
      ,amt_offer_sum_insured
      ,offer_coverage_hash  FROM silver.offer_silver""",connection)
    # split ids in two parts
    df[['id_part1','id_part2']] = df.id_offer.apply(lambda x: pd.Series([str(x)[:14],str(x)[16:]]))
    # cast id_part2 to int if populetade or replace by 0
    df["id_part2"] =  df["id_part2"].apply(lambda x: int(x.lstrip('0')) if len(x)>0 else 0 )
    df["dt_offer_date"] = pd.to_datetime(df["dt_offer_date"])
    # current timestamp minus the interval to get treshold datetime we are interesteed in
    now = pd.Timestamp.now() 
    datetime_cutoff = now - pd.Timedelta(days=1) 
    
    df = df[df["dt_offer_date"] >= datetime_cutoff]
    # after filtering large part of data away, idetify and return only newest offer records per infered offer id
    df = (
    df.loc[
        df.groupby(["id_part1"])["id_part2"]
               .idxmax()
    ]
    .reset_index(drop=True)
    )
    print(df)
    

               id_offer  id_customer offer_product_code  \
0    O-2025-0000006-D01       100774                PET   
1    O-2025-0000039-D01       100872                PET   
2        O-2025-0000056       100009               AUTO   
3        O-2025-0000120       100198             TRAVEL   
4    O-2025-0000133-D01       100167                PET   
..                  ...          ...                ...   
448  O-2025-0007859-D01       100349               AUTO   
449  O-2025-0007861-D01       100213          LIABILITY   
450      O-2025-0007875       100850          LIABILITY   
451  O-2025-0007921-D01       100723                PET   
452      O-2025-0007922       100547             TRAVEL   

    offer_product_variant_name           dt_offer_date  amt_offer_premium  \
0                          Cat 2025-06-22 13:44:31.647             123.46   
1                          Cat 2025-06-28 10:34:12.053              67.82   
2                         MTPL 2025-06-04 20:31:43.000      